# 産經新聞の記事検索結果を取得する

産経ニュースで「能登半島地震」を検索し、画面に表示される合計件数まで「もっと見る」を押して、タイトル・本文・公開日時・URLを取得します。

- 産経iDへの認証は通常のChromeで手動実行し、契約上閲覧できる本文だけを取得します。アクセス制限の回避は行いません。
- 短時間に大量アクセスしないよう、記事ごとに待機時間を設けています。
- 実行前に、利用規約・著作権・robots.txtと契約内容を確認してください。
- サイトの画面構成が変わった場合は、セレクタの調整が必要になることがあります。

In [9]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests", "beautifulsoup4", "pandas", "playwright"
])


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


0

In [10]:
import json
import re
import shutil
import subprocess
import time
from pathlib import Path
from urllib.parse import quote, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.sankei.com"
SEARCH_URL = f"{BASE_URL}/search/"
# 産経ニュース公式ページが使用しているログイン入口。認証後は検索画面へ戻す。
LOGIN_URL = (
    "https://special.sankei.com/login?return_to="
    + quote(SEARCH_URL, safe="")
)
KEYWORD = "能登半島地震"
REQUEST_INTERVAL = 1.5
TIMEOUT = 30
MAX_MORE_CLICKS = 1000  # 異常時の無限ループ防止。通常は合計件数で先に停止します。
PER_CATEGORY_LIMIT = 1000
CATEGORIES = [
    "国内", "国際", "経済", "エンタメ", "スポーツ",
    "IT", "科学", "ライフ", "地域",
]

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})

## 通常のChromeで手動ログイン

Playwrightによる認証フォーム送信は行いません。次のセルが開く通常のChromeに産経iDとパスワードを直接入力します。ログイン後はChromeを閉じず、`DONE` と入力すると、同じChromeセッションへ接続して処理を続けます。

In [11]:
print(
    "産経iDの認証情報はノートブックへ入力しません。\n"
    "次のセルで開く通常のChromeに直接入力してください。"
)

産経iDの認証情報はノートブックへ入力しません。
次のセルで開く通常のChromeに直接入力してください。


In [12]:
from playwright.async_api import async_playwright


SANKEI_PROFILE_DIR = Path(".sankei_chrome_profile").resolve()
SANKEI_CDP_PORT = 9222
SANKEI_CDP_URL = f"http://127.0.0.1:{SANKEI_CDP_PORT}"


def _find_chrome_executable():
    candidates = [
        Path("/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"),
        Path.home() / "Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    for command in ["google-chrome", "google-chrome-stable", "chromium"]:
        executable = shutil.which(command)
        if executable:
            return executable
    raise FileNotFoundError("Google Chromeの実行ファイルが見つかりません。")


def _manual_login_in_regular_chrome():
    """デバッグ接続可能な通常Chromeを開き、閉じずに手動ログインする。"""
    SANKEI_PROFILE_DIR.mkdir(parents=True, exist_ok=True)
    chrome_process = subprocess.Popen([
        _find_chrome_executable(),
        f"--user-data-dir={SANKEI_PROFILE_DIR}",
        "--no-first-run",
        "--no-default-browser-check",
        "--disable-background-mode",
        f"--remote-debugging-port={SANKEI_CDP_PORT}",
        LOGIN_URL,
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(
        "通常のChromeを開きました。産経iDへ手動でログインしてください。\n"
        "ログイン後、産経ニュースの検索ページが表示されても、"
        "このChromeは閉じないでください。"
    )
    while True:
        confirmation = input(
            "専用Chromeを開いたまま DONE と入力してください: "
        ).strip().upper()
        if confirmation != "DONE":
            print("ログイン完了後に DONE と入力してください。")
            continue
        break
    return chrome_process


async def _mark_search_result_root(page, keyword):
    """「検索結果」と合計件数を含む最小のDOM領域に印を付ける。"""
    return await page.evaluate(
        """(keyword) => {
            document.querySelectorAll('[data-codex-search-root]').forEach(
                (el) => el.removeAttribute('data-codex-search-root')
            );
            const candidates = [...document.querySelectorAll('main section, main div')]
                .map((el) => {
                    const text = (el.innerText || '').replace(/\\s+/g, ' ').trim();
                    const links = [...el.querySelectorAll('a[href*="/article/"]')]
                        .filter((a) => /\\/article\\/[0-9]{8}-[A-Z0-9]+\\//.test(a.href));
                    const hasResultLabel = /検索結果|検索した結果|件見つか/.test(text);
                    const hasCount = /[0-9０-９][0-9０-９,，]*\\s*件/.test(text);
                    return {el, text, links, hasResultLabel, hasCount};
                })
                .filter((x) => x.links.length > 0 && x.hasResultLabel && x.hasCount);
            if (!candidates.length) return null;
            candidates.sort((a, b) => {
                const aPenalty = a.text.includes('会員限定記事') ? 1000000 : 0;
                const bPenalty = b.text.includes('会員限定記事') ? 1000000 : 0;
                return (a.el.querySelectorAll('*').length + aPenalty)
                     - (b.el.querySelectorAll('*').length + bPenalty);
            });
            const root = candidates[0].el;
            root.setAttribute('data-codex-search-root', 'true');
            return root.innerText;
        }""",
        keyword,
    )


def _parse_total_count(text):
    normalized = str(text).translate(str.maketrans("０１２３４５６７８９，", "0123456789,"))
    patterns = [
        r"(?:検索結果|検索した結果)[^0-9]{0,30}([0-9][0-9,]*)\s*件",
        r"([0-9][0-9,]*)\s*件(?:の記事|見つか|\s*$)",
    ]
    for pattern in patterns:
        match = re.search(pattern, normalized)
        if match:
            return int(match.group(1).replace(",", ""))
    return None


async def _result_urls(page):
    return await page.locator('[data-codex-search-root="true"]').evaluate(
        """(root) => [...new Set(
            [...root.querySelectorAll('a[href*="/article/"]')]
                .map((a) => a.href.split('#')[0].split('?')[0])
                .filter((href) => /\\/article\\/[0-9]{8}-[A-Z0-9]+\\/$/.test(href))
        )]"""
    )


async def _set_category_filter(page, category):
    """9カテゴリの絞り込みを1つだけ選択する。"""
    trigger = page.get_by_text("カテゴリを絞り込む", exact=True)
    if await trigger.count() == 0:
        raise RuntimeError("カテゴリ絞り込みボタンが見つかりません。")
    await trigger.first.click()
    await page.wait_for_timeout(300)

    marked = await page.evaluate(
        """(categories) => {
            document.querySelectorAll('[data-codex-category-root]').forEach(
                (el) => el.removeAttribute('data-codex-category-root')
            );
            const candidates = [...document.querySelectorAll('form, section, div')]
                .filter((el) => {
                    const text = (el.innerText || '').replace(/\s+/g, ' ').trim();
                    return categories.every((name) => text.includes(name))
                        && text.includes('絞り込む')
                        && el.querySelectorAll('input[type=\"checkbox\"]').length >= 9;
                });
            if (!candidates.length) return false;
            candidates.sort((a, b) => a.querySelectorAll('*').length - b.querySelectorAll('*').length);
            candidates[0].setAttribute('data-codex-category-root', 'true');
            return true;
        }""",
        CATEGORIES,
    )
    if not marked:
        raise RuntimeError("カテゴリ選択パネルを特定できません。")

    root = page.locator('[data-codex-category-root="true"]')
    checkboxes = root.locator('input[type="checkbox"]')
    for index in range(await checkboxes.count()):
        checkbox = checkboxes.nth(index)
        if await checkbox.is_checked():
            await checkbox.uncheck(force=True)
    category_label = root.get_by_text(category, exact=True)
    if await category_label.count() != 1:
        raise RuntimeError(f"カテゴリ「{category}」を一意に特定できません。")
    await category_label.click()
    apply_button = root.get_by_role("button", name="絞り込む", exact=True)
    if await apply_button.count() != 1:
        raise RuntimeError("絞り込み実行ボタンを特定できません。")
    await apply_button.click()
    await page.wait_for_timeout(1_000)


async def login_and_collect_search_urls(requests_session, keyword):
    """手動ログイン中の通常Chromeへ接続し、検索URLを集める。"""
    chrome_process = _manual_login_in_regular_chrome()
    playwright = await async_playwright().start()
    try:
        browser = await playwright.chromium.connect_over_cdp(SANKEI_CDP_URL)
    except Exception as exc:
        await playwright.stop()
        raise RuntimeError(
            f"ログイン中のChromeへ接続できませんでした: {SANKEI_CDP_URL}"
        ) from exc
    if not browser.contexts:
        await browser.close()
        await playwright.stop()
        raise RuntimeError("ログイン中Chromeのブラウザコンテキストを取得できませんでした。")
    context = browser.contexts[0]
    # 認証Cookieは維持し、古いページ資源のキャッシュだけを削除する。
    if context.pages:
        try:
            cdp_session = await context.new_cdp_session(context.pages[-1])
            await cdp_session.send("Network.clearBrowserCache")
            await cdp_session.detach()
            print("Chromeキャッシュを削除しました（ログインCookieは維持）。")
        except Exception as exc:
            print(f"注意: Chromeキャッシュを削除できませんでした: {exc}")
    # ログイン後のタブはサイレントログインで再読み込み中のことがある。
    # 既存タブを使う場合も、クエリのない検索URLへ明示的に移動する。
    page = context.pages[-1] if context.pages else await context.new_page()

    try:
        search_box_selector = 'form.sk-searchform input#field_kw[name="kw"]'
        for attempt in range(1, 4):
            try:
                await page.goto(SEARCH_URL, wait_until="commit", timeout=20_000)
            except Exception:
                if not page.url.startswith(SEARCH_URL):
                    raise
            try:
                await page.locator(search_box_selector).wait_for(
                    state="visible", timeout=15_000
                )
                break
            except Exception as exc:
                if attempt == 3:
                    raise RuntimeError(
                        "記事検索フォームを読み込めませんでした。"
                        f"現在のURL: {page.url}"
                    ) from exc
                await page.wait_for_timeout(1_000)
        print(f"産経ニュースの検索画面を確認しました: {page.url}")
        logged_in = False
        account_label = ""
        for _ in range(20):
            login_state = await page.evaluate(
                """() => {
                    const header = document.querySelector('header');
                    const headerText = (header?.innerText || '').replace(/\s+/g, ' ').trim();
                    const logoutLink = document.querySelector('a[href*="logout" i]');
                    const accountNode = document.querySelector('[class*="user-name" i], [class*="account-name" i], [class*="member-name" i], a[href*="mypage" i]');
                    return {
                        loggedIn: document.documentElement.classList.contains('st_lg')
                            || Boolean(logoutLink)
                            || headerText.includes('ログアウト'),
                        accountLabel: (accountNode?.textContent || '').trim(),
                    };
                }"""
            )
            logged_in = bool(login_state.get("loggedIn"))
            account_label = str(login_state.get("accountLabel") or "").strip()
            if logged_in:
                break
            await page.wait_for_timeout(500)
        if not logged_in:
            sankei_cookie_count = sum(
                1 for cookie in await context.cookies()
                if "sankei" in cookie.get("domain", "")
            )
            print(
                "注意: 産経サイトのログイン表示をDOMから確認できませんでした。"
                f"産経関連Cookieは{sankei_cookie_count}件あります。"
                "手動ログイン済みとして処理を続け、本文取得結果で最終確認します。"
            )
        else:
            print("産経iDのログイン状態を確認しました。")
            if account_label:
                print(f"アカウント表示: {account_label}")
        search_box = page.locator(search_box_selector)
        if await search_box.count() != 1:
            raise RuntimeError("産経ニュースの検索入力欄を一意に特定できませんでした。")
        await search_box.fill(keyword)
        print(f"検索語を入力しました: {keyword}")
        search_button = page.locator(
            'form.sk-searchform button.sch-search-button[type="submit"]'
        )
        if await search_button.count() != 1:
            raise RuntimeError("産経ニュースの検索ボタンを一意に特定できませんでした。")
        # 送信と同時にページが遷移すると、Chromeは元のJS実行環境を
        # 破棄する。これは送信成功時にも起き得るので、遷移由来の例外は
        # 成功とみなし、新しいDOMの検索結果を待つ。
        try:
            await search_button.click(no_wait_after=True, timeout=10_000)
        except Exception as exc:
            navigation_errors = (
                "Execution context was destroyed",
                "most likely because of a navigation",
                "Target page, context or browser has been closed",
            )
            if not any(message in str(exc) for message in navigation_errors):
                raise
        print("検索フォームを送信しました。検索結果を待機します。")
        await page.wait_for_timeout(1_000)

        category_results = {}
        category_totals = {}
        for category in CATEGORIES:
            print(f"\n=== カテゴリ: {category} ===")
            await _set_category_filter(page, category)

            result_text = None
            for _ in range(40):
                try:
                    result_text = await _mark_search_result_root(page, keyword)
                except Exception as exc:
                    if "Execution context was destroyed" not in str(exc):
                        raise
                if result_text:
                    break
                await page.wait_for_timeout(500)
            if not result_text:
                raise RuntimeError(f"「{category}」の検索結果が見つかりません。")

            total_count = _parse_total_count(result_text)
            if total_count is None:
                raise RuntimeError(f"「{category}」の合計件数を読み取れません。")
            target_count = min(total_count, PER_CATEGORY_LIMIT)
            urls = await _result_urls(page)
            print(f"検索結果: {total_count:,}件 / 取得目標 {target_count:,}件")

            for click_no in range(1, MAX_MORE_CLICKS + 1):
                if len(urls) >= target_count:
                    break
                root = page.locator('[data-codex-search-root="true"]')
                more_buttons = root.get_by_role("button", name="もっと見る", exact=True)
                if await more_buttons.count() == 0:
                    more_buttons = root.locator('button').filter(has_text="もっと見る")
                if await more_buttons.count() != 1:
                    raise RuntimeError(
                        f"「{category}」{len(urls):,}/{target_count:,}件で「もっと見る」を特定できません。"
                    )
                before = len(urls)
                await more_buttons.click()
                for _ in range(40):
                    await page.wait_for_timeout(250)
                    result_text = await _mark_search_result_root(page, keyword)
                    if result_text:
                        urls = await _result_urls(page)
                    if len(urls) > before:
                        break
                if len(urls) <= before:
                    raise RuntimeError(f"「{category}」で追加結果が読み込まれません。")
                print(f"{category} もっと見る {click_no}: {len(urls):,}/{target_count:,}件")

            category_results[category] = urls[:target_count]
            category_totals[category] = total_count
            print(f"{category}: {len(category_results[category]):,}件取得")

        # ログイン済みCookieを本文取得用requests.Sessionへ移す。
        for cookie in await context.cookies():
            if "sankei" in cookie.get("domain", ""):
                requests_session.cookies.set(
                    cookie["name"], cookie["value"],
                    domain=cookie.get("domain"), path=cookie.get("path", "/"),
                )
        category_url_rows = [
            {"category": category, "url": url}
            for category, urls in category_results.items()
            for url in urls
        ]
        return category_url_rows, category_totals
    finally:
        await browser.close()
        await playwright.stop()


category_url_rows, category_totals = await login_and_collect_search_urls(
    session, KEYWORD
)
print(f"カテゴリ別URLの収集完了: {len(category_url_rows):,}件")

通常のChromeを開きました。産経iDへ手動でログインしてください。
ログイン後、産経ニュースの検索ページが表示されても、このChromeは閉じないでください。
Chromeキャッシュを削除しました（ログインCookieは維持）。
ログイン後の検索タブを再利用します: https://www.sankei.com/search/?363080


TimeoutError: Locator.wait_for: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("main form[aria-label=\"記事検索\"] input[name=\"kw\"]")


In [ ]:
def _news_article_json_ld(soup):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        items = data if isinstance(data, list) else [data]
        for item in items:
            if isinstance(item, dict) and item.get("@type") in {"NewsArticle", "Article"}:
                return item
    return {}


def fetch_article(url):
    response = session.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    metadata = _news_article_json_ld(soup)

    title = str(metadata.get("headline") or "").strip()
    if not title:
        heading = soup.select_one("main h1, article h1, h1")
        title = heading.get_text(" ", strip=True) if heading else ""

    published_at = str(metadata.get("datePublished") or "").strip()
    if not published_at:
        published = soup.select_one('time[datetime], meta[property="article:published_time"]')
        if published:
            published_at = published.get("datetime") or published.get("content") or ""

    paragraphs = []
    for node in soup.select(".article-body p.article-text"):
        text = node.get_text(" ", strip=True)
        if text and text not in paragraphs:
            paragraphs.append(text)
    body = "\n".join(paragraphs)

    article_body = soup.select_one(".article-body")
    article_body_text = article_body.get_text(" ", strip=True) if article_body else ""
    paywall_signals = (
        "続きを読むには", "会員限定記事", "有料会員記事",
        "ログインして続きを読む", "会員登録して続きを読む",
    )
    body_is_excerpt = any(signal in article_body_text for signal in paywall_signals)
    if article_body and article_body.select_one('[class*="paywall"], [id*="piano"]'):
        body_is_excerpt = True

    return {
        "title": title,
        "body": body,
        "published_at": published_at,
        "url": url,
        "body_is_excerpt": body_is_excerpt,
        "extraction_error": "" if body else "本文を取得できませんでした",
    }


def collect_articles(urls):
    records = []
    for index, url in enumerate(urls, start=1):
        try:
            record = fetch_article(url)
        except Exception as exc:
            record = {
                "title": "", "body": "", "published_at": "", "url": url,
                "body_is_excerpt": False,
                "extraction_error": f"{type(exc).__name__}: {exc}",
            }
        records.append(record)
        print(f"本文 {index:,}/{len(urls):,}: {record['title'] or url}")
        if index < len(urls):
            time.sleep(REQUEST_INTERVAL)
    return records

In [ ]:
# 同じ記事が複数カテゴリに現れても、本文へのアクセスは1回だけにする。
unique_urls = list(dict.fromkeys(row["url"] for row in category_url_rows))
unique_articles = collect_articles(unique_urls)
article_by_url = {article["url"]: article for article in unique_articles}

articles = []
for row in category_url_rows:
    article = dict(article_by_url[row["url"]])
    article["カテゴリ"] = row["category"]
    articles.append(article)

df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

for category in CATEGORIES:
    expected = min(category_totals[category], PER_CATEGORY_LIMIT)
    actual = int((df["カテゴリ"] == category).sum())
    assert actual == expected, (
        f"{category}: 期待 {expected:,}件に対し、データは {actual:,}件です。"
    )
print(df["カテゴリ"].value_counts().reindex(CATEGORIES))
df

In [ ]:
output_path = Path("sankei_能登半島地震.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")